<a href="https://colab.research.google.com/github/nguyenanhtienabcd/AIO2024_EXERCISE/blob/feature%2FMODULE10-WEEK1/m10w01_ex3_LoRA_QLoRA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -qq --upgrade pip
!pip install -qq --upgrade peft transformers accelerate bitsandbytes datasets trl huggingface_hub evaluate

In [ ]:
from google.colab import userdata
from huggingface_hub import login

# lấy token account của mình
login(token=userdata.get('your API'))

In [ ]:
import os
# Chỉ định mô hình chỉ sử dụng GPU số 0, giúp kiểm soát và tối ưu tài nguyên khi training với LoRA hoặc bất kỳ mô hình lớn nào
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

import torch
import numpy as np
import evaluate

from peft import PeftModel, PeftConfig, LoraConfig, TaskType, get_peft_model, get_peft_config
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers import DataCollatorForLanguageModeling, Trainer, TrainingArguments
from datasets import load_dataset
from trl import SFTTrainer
import warnings
from transformers import DataCollatorWithPadding
from typing import Any, Dict, List


warnings.filterwarnings("ignore")

## Hyperparameters

In [ ]:
base_model_id = "meta-llama/Llama-3.2-1B-Instruct"
cache_dir = "./cache"

MAX_TRAIN_STEPS = 5_000             # Tổng số bước huấn luyện (có thể thay thế epochs)
NUM_EVAL_STEPS = 500               # Số bước giữa mỗi lần đánh giá và lưu checkpoint
MAX_TRAIN_SAMPLES = 20_000         # Số mẫu tối đa dùng để train (giới hạn nếu tập quá lớn)
MAX_EVAL_SAMPLES = 2_000           # Số mẫu tối đa dùng để evaluate

training_args = TrainingArguments(
    output_dir="./output",                    # Nơi lưu mô hình đã huấn luyện
    per_device_train_batch_size=4,            # Batch size trên mỗi GPU trong training
    per_device_eval_batch_size=8,             # Batch size trên mỗi GPU trong eval
    logging_dir="./logs",                     # Nơi lưu logs (cho TensorBoard, v.v.)
    logging_steps=10,                         # Log mỗi 10 bước
    save_steps=NUM_EVAL_STEPS,                # Lưu checkpoint mỗi 500 bước
    max_steps=MAX_TRAIN_STEPS,                # Giới hạn tổng số bước train
    eval_steps=NUM_EVAL_STEPS,                # Đánh giá mỗi 500 bước
    eval_strategy="steps",                   # Chọn đánh giá theo bước (có thể là "epoch")
    overwrite_output_dir=True,                # Ghi đè thư mục output nếu đã tồn tại
    save_total_limit=2,                       # Chỉ giữ lại 2 checkpoint gần nhất
    report_to="none",                        # Không dùng hệ thống log bên ngoài như wandb
    push_to_hub=False,                        # Không đẩy mô hình lên HuggingFace Hub
    logging_first_step=True,                  # Log ngay từ bước đầu tiên
    remove_unused_columns=False               # Không tự động xóa các cột dữ liệu không dùng
)

## Load model

In [ ]:
# Cấu hình mô hình để thực hiện sinh văn bản (CausalLM = Causal Language Modeling) – phù hợp với fine-tuning hoặc inference dạng GPT.
base_model = AutoModelForCausalLM.from_pretrained(base_model_id, trust_remote_code=True, torch_dtype=torch.bfloat16, cache_dir=cache_dir)
tokenizer = AutoTokenizer.from_pretrained(base_model_id, trust_remote_code=True, cache_dir=cache_dir)

base_model

In [ ]:
base_model = base_model.to('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
# kiểm tra cà gán pad token
if tokenizer.pad_token is None or tokenizer.pad_token_id is None:
    print("Pad token is not set. Setting it to EOS token.")
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id
else:
    print(f'Pad token: {tokenizer.pad_token}')
    print(f'Pad token id: {tokenizer.pad_token_id}')

print(f'EOS token: {tokenizer.eos_token}')
print(f'EOS token id: {tokenizer.eos_token_id}')

### template định dạng hội thoại

In [ ]:
if tokenizer.chat_template is None:
    tokenizer.chat_template = """{{- bos_token }}
{%- if not date_string is defined %}
    {%- if strftime_now is defined %}{%- set date_string = strftime_now("%d %b %Y") %}{%- else %}{%- set date_string = "26 Jul 2024" %}{%- endif %}
{%- endif %}

{#- This block extracts the system message, so we can slot it into the right place. #}
{%- if messages[0]['role'] == 'system' %}
    {%- set system_message = messages[0]['content']|trim %}
    {%- set messages = messages[1:] %}
{%- else %}
    {%- set system_message = "" %}
{%- endif %}

{#- System message #}
{{- "<|start_header_id|>system<|end_header_id|>\n\n" }}
{{- "Cutting Knowledge Date: December 2023\n" }}
{{- "Today Date: " + date_string + "\n\n" }}
{{- system_message }}
{{- "<|eot_id|>" }}

{%- for message in messages %}
    {{- '<|start_header_id|>' + message['role'] + '<|end_header_id|>\n\n'+ message['content'] | trim + '<|eot_id|>' }}
{%- endfor %}
{%- if add_generation_prompt %}
    {{- '<|start_header_id|>assistant<|end_header_id|>\n\n' }}
{%- endif %}
"""

## Load and Apply LoRA

In [ ]:
peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    inference_mode=False,
    r=8,
    lora_alpha=32,
    lora_dropout=0.1
)

In [ ]:
peft_model = get_peft_model(base_model, peft_config)
peft_model.print_trainable_parameters()
peft_model

## Load Dataset and Format

In [ ]:
dataset = load_dataset("uitnlp/vietnamese_students_feedback", cache_dir=cache_dir)

for split in dataset:
    if split == "train":
        MAX_TRAIN_SAMPLES = min(MAX_TRAIN_SAMPLES, len(dataset[split]))
        dataset[split] = dataset[split].select(range(MAX_TRAIN_SAMPLES))
    else:
        MAX_EVAL_SAMPLES = min(MAX_EVAL_SAMPLES, len(dataset[split]))
        dataset[split] = dataset[split].select(range(MAX_EVAL_SAMPLES))
    print(f"{split}: {len(dataset[split])}")

In [ ]:
print(f'A sample from the train set: {dataset["train"][0]}')

In [ ]:
label_set = set([item["sentiment"] for split in dataset for item in dataset[split]])
label_set

In [ ]:
all_labels = dataset['train'].features['sentiment'].names
print(f'There are {len(all_labels)} labels in the dataset, including {all_labels}')

label2id = {label: i for i, label in enumerate(all_labels)}
id2label = {i: label for i, label in enumerate(all_labels)}

print(f'label2id: {label2id}')
print(f'id2label: {id2label}')

In [ ]:
USER_PROMPT_TEMPLATE = """Predict the sentiment of the following input sentence.
The response must begin with "Sentiment: ", followed by one of these keywords: "positive", "negative", or "neutral", to reflect the sentiment of the input sentence.

Sentence: {input}"""

def tokenize_function(examples):
    # Khởi tạo dictionary chứa kết quả
    results = {
        "input_ids": [],          # Danh sách token đầu vào
        "labels": [],             # Danh sách token đích để tính loss
        "attention_mask": [],     # Mặt nạ chú ý (1 cho token thật, 0 nếu là padding)
    }

    # Lặp qua từng ví dụ trong batch
    for i in range(len(examples['sentence'])):
        cur_input = examples['sentence'][i]        # Câu đầu vào (text)
        cur_output_id = examples['sentiment'][i]   # Nhãn đầu ra (ID: 0, 1, 2)

        # Áp dụng template để tạo câu hỏi đầu vào
        cur_prompt = USER_PROMPT_TEMPLATE.format(input=cur_input)

        # Chuyển nhãn ID thành chuỗi cảm xúc: "positive", "neutral", "negative"
        cur_output = id2label[cur_output_id]

        # Định nghĩa prompt hội thoại (chat): role = system và user
        input_messages = [
            {"role": "system", "content": "You are a helpful assistant. You must fulfill the user request."},
            {"role": "user", "content": cur_prompt},
        ]

        # Thêm câu trả lời mẫu từ assistant (là nhãn đúng)
        input_output_messages = input_messages + [
            {"role": "assistant", "content": f"Sentiment: {cur_output}"}
        ]

        # Mã hóa đoạn prompt (chỉ input) và thêm tín hiệu để mô hình sinh tiếp
        input_prompt_tokenized = tokenizer.apply_chat_template(
            conversation=input_messages,
            return_tensors="pt",
            add_generation_prompt=True  # Thêm cue cho mô hình biết sinh tiếp
        )[0]  # Lấy tensor từ batch = 1

        # Mã hóa toàn bộ đoạn chat gồm cả phản hồi đúng (input + output)
        input_output_prompt_tokenized = tokenizer.apply_chat_template(
            conversation=input_output_messages,
            return_tensors="pt"
        )[0]

        # Đầu vào mô hình là toàn bộ câu hỏi + câu trả lời
        input_ids = input_output_prompt_tokenized

        # Với phần nhãn, ta cần mask phần câu hỏi bằng -100 (bỏ qua khi tính loss)
        label_ids = torch.cat([
            torch.full_like(input_prompt_tokenized, fill_value=-100),  # phần prompt (bỏ qua)
            input_output_prompt_tokenized[len(input_prompt_tokenized):]  # phần output (tính loss)
        ])

        # Đảm bảo chiều dài input và label phải giống nhau
        assert len(input_ids) == len(label_ids)

        # Thêm vào kết quả
        results["input_ids"].append(input_ids)
        results["labels"].append(label_ids)
        results["attention_mask"].append(torch.ones_like(input_ids))  # Tất cả token đều là 1 (chưa padding)

    return results


col_names = dataset['train'].column_names
tokenized_dataset = dataset.map(
    tokenize_function,          # Hàm biến đổi từng phần tử hoặc batch
    batched=True,               # Áp dụng theo batch thay vì từng dòng
    remove_columns=col_names,   # Xóa cột gốc ('sentence', 'sentiment')
    num_proc=os.cpu_count(),    # Chạy song song tối đa theo số lõi CPU
)
tokenized_dataset

In [ ]:
print(tokenized_dataset['train'][0])
print(tokenizer.decode(tokenized_dataset['train'][0]['input_ids'], skip_special_tokens=False))

## Custom data collator

Trong quá trình huấn luyện, các mẫu dữ liệu sẽ có độ dài không đồng đều. Do đó, ta cần phải
sử dụng padding token để các câu có độ dài bằng nhau

In [ ]:
class RightPaddingDataCollator(DataCollatorWithPadding):
    """The default data collator pads only inputs, not including the labels."""

    # kế thừa, sử dụng thuộc tính từ lớp cha tokenizer, max_length
    def __init__(self, tokenizer, max_length: int = 1024):
        super().__init__(tokenizer, max_length=max_length)

    # sử dụng hàm call để xem đối tượng như một hàm (class as a function)
    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, Any]:
        input_ids, labels, attention_mask = [], [], []
        max_batch_len = max(len(f["input_ids"]) for f in features)

        for sample in features:
            # Convert to torch tensors
            cur_input_ids = torch.tensor(sample["input_ids"], dtype=torch.long)
            cur_labels = torch.tensor(sample["labels"], dtype=torch.long)
            cur_attention_mask = torch.ones_like(cur_input_ids)

            # Next, we pad the inputs and labels to the maximum length within the batch
            pad_token_id = self.tokenizer.pad_token_id
            padding_length = max_batch_len - len(cur_input_ids)
            cur_input_ids = torch.cat([cur_input_ids, torch.full((padding_length,), fill_value=pad_token_id, dtype=torch.long)])
            cur_labels = torch.cat([cur_labels, torch.full((padding_length,), fill_value=-100, dtype=torch.long)])
            cur_attention_mask = torch.cat([cur_attention_mask, torch.zeros((padding_length,), dtype=torch.long)])

            # Truncate the inputs and labels to the maximum length
            cur_input_ids = cur_input_ids[:max_batch_len]
            cur_labels = cur_labels[:max_batch_len]
            cur_attention_mask = cur_attention_mask[:max_batch_len]

            # Append to the return lists
            input_ids.append(cur_input_ids)
            labels.append(cur_labels)
            attention_mask.append(cur_attention_mask)

        # Return formatted batch.
        return {
            "input_ids": torch.stack(input_ids),
            "labels": torch.stack(labels),
            "attention_mask": torch.stack(attention_mask)
        }


data_collator = RightPaddingDataCollator(tokenizer)

In [ ]:
# xây dựng các matrix
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")
precision_metric = evaluate.load("precision")
recall_metric = evaluate.load("recall")

In [ ]:
# logit là đầu ra của mô hình logits = model() => logits = (tensor([[...]]), hidden_states, attentions)
def preprocess_logits_for_metrics(logits, labels):
    if isinstance(logits, tuple):
        logits = logits[0]
    return logits.argmax(dim=-1)


def compute_metrics(eval_preds):
    preds, labels = eval_preds

    # Nếu mô hình trả về tuple (logits, ...), chỉ lấy logits
    if isinstance(preds, tuple):
        preds = preds[0]

    # Tìm vị trí kết thúc phần prompt bị mask (-100)
    idx = 0
    for i in range(len(labels[0])):
        if labels[0][i] == -100:
            idx = i
        else:
            break

    # Cắt bỏ phần đầu (prompt) trong output dự đoán
    preds = preds[:, idx:]

    # Thay -100 trong dự đoán bằng pad_token_id để có thể decode
    preds = np.where(preds != -100, preds, tokenizer.pad_token_id)

    # Tách từng dòng để xử lý riêng
    processed_preds = []
    for pred in preds:
        # Dừng lại ở token <eos> đầu tiên nếu có
        end_pred_idx = np.where(pred == tokenizer.eos_token_id)[0]
        if len(end_pred_idx) > 0:
            end_pred_idx = end_pred_idx[0]
            processed_preds.append(pred[:end_pred_idx])
        else:
            processed_preds.append(pred)

    # Giải mã các dự đoán thành chuỗi văn bản
    decoded_preds = tokenizer.batch_decode(processed_preds, skip_special_tokens=True)

    # Giải mã nhãn (labels) sau khi thay -100 bằng pad_token
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    # chuyển các chuỗi token thành văn bản gốc
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Chuyển text → ID bằng label2id
    int_preds, int_labels = [], []
    for p, l in zip(decoded_preds, decoded_labels):
        l = l.split(":")[-1].strip()
        cur_label_id = label2id[l]  # Lấy ID từ label thật
        int_labels.append(cur_label_id)

        try:
            p = p.split(":")[-1].strip()
            cur_pred_id = label2id[p]  # Lấy ID từ dự đoán nếu đúng format
        except Exception as e:
            # Nếu lỗi → chọn nhãn sai lệch (để bị tính sai)
            cur_pred_id = (cur_label_id + 1) % len(label2id)
        int_preds.append(cur_pred_id)

    # Tính các chỉ số đánh giá
    accuracy_results = accuracy_metric.compute(predictions=int_preds, references=int_labels)
    f1_results = f1_metric.compute(predictions=int_preds, references=int_labels, average="macro")
    precision_results = precision_metric.compute(predictions=int_preds, references=int_labels, average="macro")
    recall_results = recall_metric.compute(predictions=int_preds, references=int_labels, average="macro")

    # mục đích gộp tất cả các dict nhỏ thành một dict lớn
    return {
        **accuracy_results,
        **f1_results,
        **precision_results,
        **recall_results
    }

## Train the Model

In [ ]:
trainer = SFTTrainer(
    model=peft_model,
    args=training_args,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['validation'],
    preprocess_logits_for_metrics=preprocess_logits_for_metrics,
    compute_metrics=compute_metrics,
    processing_class=tokenizer,
    data_collator=data_collator,
)
trainer.train()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

train_stats = pd.DataFrame(trainer.state.log_history)
train_stats = train_stats[:-1]
train_stats

## Inference Pipeline

In [ ]:
def inference(model, tokenizer, input_sentence):
    tokenizer.pad_token_id = tokenizer.eos_token_id

    user_prompt = USER_PROMPT_TEMPLATE.format(input=input_sentence)
    messages = [
        {"role": "system", "content": "You are a helpful assistant. You must fulfill the user request."},
        {"role": "user", "content": user_prompt},
    ]
    input_prompt = tokenizer.apply_chat_template(conversation=messages, add_generation_prompt=True, tokenize=False)
    inputs = tokenizer(input_prompt, return_tensors="pt", add_special_tokens=False)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    output_ids = model.generate(**inputs, max_new_tokens=16, pad_token_id=tokenizer.eos_token_id)
    output_ids = output_ids[:, inputs['input_ids'][0].shape[-1]:output_ids.shape[-1]]
    results = tokenizer.batch_decode(output_ids, skip_special_tokens=True)
    return results[0]

def batch_inference(model, tokenizer, input_sentences):
    tokenizer.padding_side = "left"
    tokenizer.pad_token_id = tokenizer.eos_token_id

    user_prompts = [USER_PROMPT_TEMPLATE.format(input=input_sentence) for input_sentence in input_sentences]
    messages_list = [
        [
            {"role": "system", "content": "You are a helpful assistant. You must fulfill the user request."},
            {"role": "user", "content": user_prompt},
        ]
        for user_prompt in user_prompts
    ]
    input_prompts = [tokenizer.apply_chat_template(conversation=messages, add_generation_prompt=True, tokenize=False) for messages in messages_list]

    inputs = tokenizer(input_prompts, return_tensors="pt", padding=True, add_special_tokens=False)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    output_ids = model.generate(**inputs, max_new_tokens=16, pad_token_id=tokenizer.eos_token_id)
    output_ids = output_ids[:, inputs['input_ids'][0].shape[-1]:output_ids.shape[-1]]
    results = tokenizer.batch_decode(output_ids, skip_special_tokens=True)
    return results

In [ ]:
inference(peft_model, tokenizer, "The weather is nice today."), inference(peft_model, tokenizer, "I love this product.")

In [ ]:
batch_inference(peft_model, tokenizer, ["I love this product.", "I hate this product. It is because the quality is extremely bad."])

In [ ]:
batch_inference(peft_model, tokenizer, ["Môn học này quá khó để học", "Thầy dạy hay, dễ hiểu"])

In [ ]:
train_df = train_stats[train_stats['step'] % 500 == 0].copy()
train_df = train_df[train_df['loss'].notna()]
eval_df = train_stats[train_stats['step'] % 500 == 0].copy()
eval_df = eval_df[eval_df['eval_loss'].notna()]

# filtered_df = train_stats[train_stats['step'] % 500 == 0].copy()
# filtered_df = filtered_df.fillna(method='ffill')
train_step = train_df['step'].to_numpy()
train_loss = train_df['loss'].to_numpy()
evval_loss = eval_df['eval_loss'].to_numpy()
eval_accuracy = eval_df['eval_accuracy'].to_numpy()
eval_f1 = eval_df['eval_f1'].to_numpy()

fig, ax = plt.subplots(1, 2, figsize=(12, 4), dpi=1000)

# plot 1 is loss and eval_loss
ax[0].plot(train_step, train_loss, label='Training Loss', marker='o')
ax[0].plot(train_step, evval_loss, label='Validation Loss', marker='o')
ax[0].set_xlabel('Step')
ax[0].set_ylabel('Loss')
ax[0].set_title('Training and Validation Loss Over Time')
ax[0].legend()
ax[0].grid(True)

# plot 2 is eval_accuracy and eval_f1
ax[1].plot(train_step, eval_accuracy, label='Validation Accuracy', marker='o')
ax[1].plot(train_step, eval_f1, label='Validation F1', marker='o')
ax[1].set_xlabel('Step')
ax[1].set_ylabel('Score')
ax[1].set_title('Validation Accuracy and F1 Over Time')
ax[1].legend()
ax[1].grid(True)

# save figure to train_results.pdf
plt.savefig('train_results.pdf')

In [ ]:
# Evaluate the model on the test set
trainer.evaluate(tokenized_dataset['test'])

In [ ]:
# Push to the hub
hub_model_name = "<username>/<model_name>"
hub_model_name = "tmnam20/Llama-3.2-1B-LoRA-vsf"
peft_model.push_to_hub(hub_model_name)
tokenizer.push_to_hub(hub_model_name)

# or save to local directory
# peft_model.save_pretrained("./peft-lora-causal-lm-1b")
# tokenizer.save_pretrained("./peft-lora-causal-lm-1b")

## Evaluate the models using zero-, one-, and few-shot learning

In [ ]:
def evaluate_zero_shot(model, tokenizer, eval_dataset, batch_size=8):
    model.eval()  # Set model to evaluation mode
    all_predictions = []
    all_labels = []

    print_example = True

    # Process dataset in batches
    for i in range(0, len(eval_dataset), batch_size):
        batch = eval_dataset[i:i + batch_size]

        # Get predictions for batch
        predictions = batch_inference(model, tokenizer, batch['sentence'])
        if print_example:
            print_example = False
            for sentence, prediction, label in zip(batch['sentence'], predictions, batch['sentiment']):
                print(f"Sentence: {sentence}")
                print(f"Model Output: {prediction}")
                print(f"Label: {id2label[label]}")
                print()

        # Convert text predictions to label ids
        pred_ids = []
        true_labels = batch['sentiment']

        for p, l in zip(predictions, true_labels):
            try:
                label_id = l
                p = p.split(":")[-1].strip()
                pred_id = label2id[p]
            except Exception as e:
                pred_id = (l + 1) % len(label2id) # Choose the next label, ensure the prediction is marked as incorrect
            pred_ids.append(pred_id)

        all_predictions.extend(pred_ids)
        all_labels.extend(true_labels)

    accuracy_metric = evaluate.load("accuracy")
    f1_metric = evaluate.load("f1")
    precision_metric = evaluate.load("precision")
    recall_metric = evaluate.load("recall")

    # Calculate metrics
    metrics = {
        'accuracy': accuracy_metric.compute(predictions=all_predictions, references=all_labels),
        'f1': f1_metric.compute(predictions=all_predictions, references=all_labels, average='macro'),
        'precision': precision_metric.compute(predictions=all_predictions, references=all_labels, average='macro'),
        'recall': recall_metric.compute(predictions=all_predictions, references=all_labels, average='macro')
    }

    # Combine all metrics into single dict
    results = {}
    for metric_name, metric_dict in metrics.items():
        results.update(metric_dict)

    return results

In [ ]:
# Example usage
zero_shot_results = evaluate_zero_shot(
    model=peft_model,
    tokenizer=tokenizer,
    eval_dataset=dataset['test'],
    batch_size=8
)

print("Zero-shot evaluation results:")
for metric_name, value in zero_shot_results.items():
    print(f"{metric_name}: {value:.4f}")

In [ ]:
USER_FEWSHOT_PROMPT_TEMPLATE = """Predict the sentiment of the following input sentence.
The response must begin with "Sentiment: ", followed by one of these keywords: "positive", "negative", or "neutral", to reflect the sentiment of the input sentence.

Here are a few examples:

{few_shot_examples}

Sentence: {input}"""

def evaluate_few_shot(model, tokenizer, eval_dataset, few_shot_examples, batch_size=8, print_example=False):
    model.eval()  # Đặt mô hình ở chế độ đánh giá (không dropout, không update weight)

    all_predictions = []  # Lưu dự đoán cuối cùng (label_id)
    all_labels = []       # Lưu nhãn thật (label_id)

    # === 1. Tạo đoạn ví dụ (few-shot) đưa vào prompt ===
    formatted_few_shot_examples = ""
    for i, example in enumerate(few_shot_examples):
        # Gộp nhiều ví dụ thành định dạng: Sentence + Sentiment
        formatted_few_shot_examples += f"Sentence: {example['sentence']}\nSentiment: {id2label[example['sentiment']]}\n"
        if i < len(few_shot_examples) - 1:
            formatted_few_shot_examples += "\n"  # Thêm dòng trống giữa các ví dụ

    # === 2. Duyệt qua tập đánh giá theo batch ===
    for i in range(0, len(eval_dataset), batch_size):
        batch = eval_dataset[i:i + batch_size]

        # Tạo prompt cho mỗi câu trong batch, kèm theo ví dụ few-shot
        user_prompts = [
            USER_FEWSHOT_PROMPT_TEMPLATE.format(
                input=sentence,
                few_shot_examples=formatted_few_shot_examples
            )
            for sentence in batch['sentence']
        ]

        # Tạo prompt hội thoại dạng chat
        messages_list = [
            [
                {"role": "system", "content": "You are a helpful assistant. You must fulfill the user request."},
                {"role": "user", "content": user_prompt},
            ]
            for user_prompt in user_prompts
        ]

        # Áp dụng chat template để định dạng theo LLaMA / GPT-style
        input_prompts = [
            tokenizer.apply_chat_template(
                conversation=messages,
                add_generation_prompt=True,
                tokenize=False
            )
            for messages in messages_list
        ]

        # Mã hóa toàn bộ batch → tensor (có padding trái)
        inputs = tokenizer(
            input_prompts,
            return_tensors="pt",
            padding=True,
            add_special_tokens=False
        )
        inputs = {k: v.to(model.device) for k, v in inputs.items()}

        # Dự đoán bằng mô hình, giới hạn 16 token sinh ra
        output_ids = model.generate(
            **inputs,
            max_new_tokens=16,
            pad_token_id=tokenizer.eos_token_id
        )

        # Cắt phần đầu vào → chỉ giữ phần sinh mới
        output_ids = output_ids[:, inputs['input_ids'][0].shape[-1]:output_ids.shape[-1]]

        # Giải mã kết quả (list chuỗi)
        predictions = tokenizer.batch_decode(output_ids, skip_special_tokens=True)

        # In thử một ví dụ (nếu được yêu cầu)
        if print_example:
            print_example = False
            print(f"### Prompt:\n{user_prompts[0]}")
            print(f"### Model Output:\n{predictions[0]}")
            print(f"### Label:\n{id2label[batch['sentiment'][0]]}")
            print()

        # === 3. Xử lý kết quả và đánh giá ===
        pred_ids = []
        true_labels = batch['sentiment']  # Nhãn thật của batch

        for p, l in zip(predictions, true_labels):
            try:
                # Tách phần nhãn từ output: "Sentiment: positive" → "positive"
                p = p.split(":")[-1].strip()
                pred_id = label2id[p]  # Chuyển từ text → ID
            except Exception:
                # Nếu lỗi (output không hợp lệ), chọn nhãn khác để tính sai
                pred_id = (l + 1) % len(label2id)
            pred_ids.append(pred_id)

        all_predictions.extend(pred_ids)
        all_labels.extend(true_labels)

    # === 4. Tính các chỉ số đánh giá ===
    accuracy_metric = evaluate.load("accuracy")
    f1_metric = evaluate.load("f1")
    precision_metric = evaluate.load("precision")
    recall_metric = evaluate.load("recall")

    metrics = {
        'accuracy': accuracy_metric.compute(predictions=all_predictions, references=all_labels),
        'f1': f1_metric.compute(predictions=all_predictions, references=all_labels, average='macro'),
        'precision': precision_metric.compute(predictions=all_predictions, references=all_labels, average='macro'),
        'recall': recall_metric.compute(predictions=all_predictions, references=all_labels, average='macro')
    }

    # Gộp tất cả các metric vào 1 dict kết quả
    results = {}
    for metric_name, metric_dict in metrics.items():
        results.update(metric_dict)

    return results  # Trả về dict chứa accuracy, f1, precision, recall


In [ ]:
# Pick a list of shot from the train set
shuffled_train_dataset = dataset['train'].shuffle()
sampled_few_shot_examples = list(shuffled_train_dataset.select(range(10)))
few_shot_result_df = pd.DataFrame(columns=['n_shots', 'accuracy', 'f1', 'precision', 'recall'])

n_shots = [1, 2, 4, 8]
for n in n_shots:
    few_shot_examples = sampled_few_shot_examples[:n]
    few_shot_results = evaluate_few_shot(
        model=base_model,
        tokenizer=tokenizer,
        eval_dataset=dataset['test'],
        few_shot_examples=few_shot_examples,
        batch_size=16,
        print_example=True,
    )
    print(f"*** Few-shot evaluation results with {n} shots:")
    for metric_name, value in few_shot_results.items():
        print(f"* {metric_name}: {value:.4f}")
    print()
    few_shot_result_df.loc[len(few_shot_result_df)] = [n, few_shot_results['accuracy'], few_shot_results['f1'], few_shot_results['precision'], few_shot_results['recall']]

few_shot_result_df = few_shot_result_df.round(4).astype(str)
few_shot_result_df['accuracy'] = few_shot_result_df['accuracy'].apply(lambda x: f"{float(x):.4f}")
few_shot_result_df['f1'] = few_shot_result_df['f1'].apply(lambda x: f"{float(x):.4f}")
few_shot_result_df['precision'] = few_shot_result_df['precision'].apply(lambda x: f"{float(x):.4f}")
few_shot_result_df['recall'] = few_shot_result_df['recall'].apply(lambda x: f"{float(x):.4f}")
few_shot_result_df['n_shots'] = few_shot_result_df['n_shots'].apply(lambda x: int(float(x)))
few_shot_result_df

In [ ]:
few_shot_result_df['accuracy'] = few_shot_result_df['accuracy'].apply(lambda x: f"{float(x):.4f}")
few_shot_result_df['f1'] = few_shot_result_df['f1'].apply(lambda x: f"{float(x):.4f}")
few_shot_result_df['precision'] = few_shot_result_df['precision'].apply(lambda x: f"{float(x):.4f}")
few_shot_result_df['recall'] = few_shot_result_df['recall'].apply(lambda x: f"{float(x):.4f}")
few_shot_result_df['n_shots'] = few_shot_result_df['n_shots'].apply(lambda x: int(float(x)))
few_shot_result_df

In [ ]:
# save few_shot_result_df to latex
few_shot_result_df.to_latex('few_shot_result_df.tex', index=False)

## Changing rank of LoRA

In [ ]:
def train_lora(base_model, tokenizer, training_args, lora_rank, dataset):
    peft_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM, inference_mode=False, r=lora_rank, lora_alpha=32, lora_dropout=0.1
    )
    cur_peft_model = get_peft_model(base_model, peft_config)
    cur_peft_model.print_trainable_parameters()

    trainer = SFTTrainer(
        model=cur_peft_model,
        args=training_args,
        train_dataset=dataset['train'],
        eval_dataset=dataset['validation'],
        preprocess_logits_for_metrics=preprocess_logits_for_metrics,
        compute_metrics=compute_metrics,
        processing_class=tokenizer,
        data_collator=data_collator,
    )
    trainer.train()
    return cur_peft_model


In [ ]:
ranks = [1, 2, 4, 8, 16, 32, 64, 128]
# ranks = [1, 2]
rank_results = pd.DataFrame(columns=['rank', 'accuracy', 'f1', 'precision', 'recall'])
for rank in ranks:
    print(f'*** Train with rank {rank}')
    cur_trained_model = train_lora(base_model, tokenizer, training_args, rank, tokenized_dataset)
    cur_results = evaluate_zero_shot(
        model=cur_trained_model,
        tokenizer=tokenizer,
        eval_dataset=dataset['test'],
        batch_size=8
    )

    # add current results to rank_results
    rank_results.loc[len(rank_results)] = [rank, cur_results['accuracy'], cur_results['f1'], cur_results['precision'], cur_results['recall']]
rank_results

In [ ]:
rank_results['accuracy'] = rank_results['accuracy'].apply(lambda x: f"{float(x):.4f}")
rank_results['f1'] = rank_results['f1'].apply(lambda x: f"{float(x):.4f}")
rank_results['precision'] = rank_results['precision'].apply(lambda x: f"{float(x):.4f}")
rank_results['recall'] = rank_results['recall'].apply(lambda x: f"{float(x):.4f}")
rank_results['rank'] = rank_results['rank'].apply(lambda x: int(float(x)))
rank_results.to_latex('rank_results.tex', index=False)
rank_results